# 03 — Grad-CAM Analysis

Visualise **what each model looks at** when making a prediction.

For each sample image we run Grad-CAM through all 5 models and overlay the
heatmap on the original X-ray.  A well-trained model should highlight
clinically relevant anatomy.  A biased model may fixate on texture or edge
artefacts introduced by style transfer.

| Model | Trained on | Bias type |
|-------|------------|-----------|
| original | real X-rays | none (baseline) |
| gb | gaussian blur | texture |
| ps | patch shuffle | texture |
| ce | canny edge | shape |
| pr | patch rotation | shape |

In [1]:
import os, sys
import numpy as np
import polars as pl
import torch
import matplotlib.pyplot as plt
import matplotlib.cm as cm
from pathlib import Path
from PIL import Image
from torchvision import transforms

_cwd = Path().resolve()
REPO_ROOT = next(
    (p for p in [_cwd, *_cwd.parents] if (p / 'pyproject.toml').exists()), _cwd
)
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

from src.models.densenet import DenseNetClassifier

DEVICE   = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
IMG_ROOT = REPO_ROOT / 'src' / 'data' / '1'
print('Device:', DEVICE)
print('Repo root:', REPO_ROOT)

: 

## 1. Load all 5 models

In [1]:
CHECKPOINTS = {
    'original': REPO_ROOT / 'results/original/best_model.pth',
    'gb':       REPO_ROOT / 'results/gb/best_model.pth',
    'ps':       REPO_ROOT / 'results/ps/best_model.pth',
    'ce':       REPO_ROOT / 'results/ce/best_model.pth',
    'pr':       REPO_ROOT / 'results/pr/best_model.pth',
}

LABEL_NAMES = [
    'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion',
    'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax',
    'Pleural Effusion', 'Pleural Other', 'Fracture', 'No Finding', 'Support Devices',
]

MODEL_LABELS = {
    'original': 'Baseline',
    'gb':       'Gaussian Blur\n(texture)',
    'ps':       'Patch Shuffle\n(texture)',
    'ce':       'Canny Edge\n(shape)',
    'pr':       'Patch Rotation\n(shape)',
}

def load_model(ckpt_path):
    model = DenseNetClassifier(num_classes=14, pretrained=False, variant='densenet121').to(DEVICE)
    ckpt  = torch.load(ckpt_path, map_location=DEVICE)
    model.load_state_dict(ckpt['model_state_dict'])
    model.eval()
    return model

models = {name: load_model(path) for name, path in CHECKPOINTS.items()}
print('Loaded:', list(models.keys()))

NameError: name 'REPO_ROOT' is not defined

## 2. Grad-CAM implementation

We hook the **last feature map** of the DenseNet backbone (`model.backbone.features`).
Grad-CAM weights each spatial location by how strongly its activations influenced
the target class score — high weight = model was paying attention there.

In [ ]:
class GradCAM:
    """Grad-CAM for DenseNet. Hooks model.backbone.features (last feature map)."""

    def __init__(self, model):
        self.model       = model
        self.activations = None
        self.gradients   = None
        model.backbone.features.register_forward_hook(self._save_activation)
        model.backbone.features.register_full_backward_hook(self._save_gradient)

    def _save_activation(self, module, input, output):
        self.activations = output.detach()

    def _save_gradient(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def __call__(self, x, class_idx):
        """Return normalised CAM heatmap (H, W) in [0, 1]."""
        self.model.zero_grad()
        output = self.model(x)           # (1, 14) logits
        output[0, class_idx].backward()  # backprop through target class only

        # Global average pool gradients over spatial dims to get channel weights
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam     = (weights * self.activations).sum(dim=1)         # (1, H, W)
        cam     = torch.relu(cam).squeeze().cpu().numpy()         # (H, W)
        cam     = (cam - cam.min()) / (cam.max() - cam.min() + 1e-8)
        return cam


grad_cams = {name: GradCAM(m) for name, m in models.items()}
print('GradCAM hooks registered for:', list(grad_cams.keys()))

## 3. Pick sample images

Select images from the original test manifest with confirmed positive labels
for high-AUROC pathologies — Edema and Pleural Effusion.

In [ ]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]

preprocess = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

def load_sample(rel_path):
    full_path = IMG_ROOT / rel_path
    pil_img   = Image.open(full_path).convert('RGB').resize((224, 224))
    tensor    = preprocess(Image.open(full_path).convert('RGB')).unsqueeze(0).to(DEVICE)
    return pil_img, tensor

test_df = pl.read_parquet(REPO_ROOT / 'src/data/test_manifest.parquet')
test_df = test_df.with_columns(
    pl.col('Path').map_elements(
        lambda p: str(Path(*Path(p).parts[1:])), return_dtype=pl.Utf8
    ).alias('rel_path')
)

edema_samples    = test_df.filter(pl.col('Edema') == 1).head(2)['rel_path'].to_list()
effusion_samples = test_df.filter(pl.col('Pleural Effusion') == 1).head(2)['rel_path'].to_list()
sample_paths     = edema_samples + effusion_samples
sample_labels    = ['Edema'] * 2 + ['Pleural Effusion'] * 2

print(f'Selected {len(sample_paths)} samples')
for p, l in zip(sample_paths, sample_labels):
    print(f'  [{l}]  {p}')

## 4. Grad-CAM grid — all models × all samples

Each **row** = one sample. Each **column** = one model.  
Red = high attention, blue = low attention.

In [ ]:
def overlay_heatmap(pil_img, cam, alpha=0.45):
    heatmap = cm.jet(cam)[:, :, :3]
    img_arr = np.array(pil_img) / 255.0
    return np.clip((1 - alpha) * img_arr + alpha * heatmap, 0, 1)


model_names = list(models.keys())
n_samples   = len(sample_paths)
n_models    = len(model_names)

fig, axes = plt.subplots(
    n_samples, n_models + 1,
    figsize=(3.5 * (n_models + 1), 3.5 * n_samples)
)

for row, (rel_path, pathology) in enumerate(zip(sample_paths, sample_labels)):
    try:
        pil_img, tensor = load_sample(rel_path)
    except FileNotFoundError:
        print(f'Image not found: {rel_path}')
        continue

    class_idx = LABEL_NAMES.index(pathology)

    # Column 0 — raw image
    axes[row, 0].imshow(pil_img, cmap='gray')
    axes[row, 0].set_title(f'Original X-ray\n[{pathology}]', fontsize=8)
    axes[row, 0].axis('off')

    # Columns 1-5 — Grad-CAM per model
    for col, name in enumerate(model_names, start=1):
        cam     = grad_cams[name](tensor, class_idx)
        overlay = overlay_heatmap(pil_img, cam)
        axes[row, col].imshow(overlay)
        axes[row, col].set_title(MODEL_LABELS[name], fontsize=7)
        axes[row, col].axis('off')

fig.suptitle('Grad-CAM: What each model attends to', fontsize=13, y=1.01)
fig.tight_layout()

out_path = REPO_ROOT / 'results/grad_cam_comparison.png'
fig.savefig(out_path, dpi=150, bbox_inches='tight')
print(f'Saved: {out_path}')
plt.show()

## 5. Predicted probability per model

For the same samples, compare each model's confidence on the target pathology.
Biased models should show lower confidence on real images.

In [ ]:
header = f"{'Image':<40} {'Pathology':<20}" + "".join(f"{n:>12}" for n in model_names)
print(header)
print("-" * len(header))

for rel_path, pathology in zip(sample_paths, sample_labels):
    try:
        _, tensor = load_sample(rel_path)
    except FileNotFoundError:
        continue

    class_idx = LABEL_NAMES.index(pathology)
    probs = []
    with torch.no_grad():
        for name in model_names:
            prob = torch.sigmoid(models[name](tensor)[0, class_idx]).item()
            probs.append(prob)

    print(f"{Path(rel_path).name:<40} {pathology:<20}" + "".join(f"{p:>12.4f}" for p in probs))

## 6. What to look for

| Observation | Interpretation |
|---|---|
| Baseline heatmap concentrated on lungs/heart | Model learned real anatomy |
| gb/ps heatmap scattered or uniform | Texture model lost spatial focus |
| ce/pr heatmap on image borders/edges | Shape model attending to contours not pathology |
| Biased model lower probability than baseline | Bias cost real diagnostic confidence |

If matching reliance > 1 (from `results/bias_eval/reliance.json`) **and** the
Grad-CAM shows attention on artefact regions rather than anatomy, you have both
quantitative and visual confirmation of the bias.